<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c05-header.png" alt="Nextia Learning · Machine Learning: From Problem to Reliable Model" width="100%">

# Solution: Slice analysis

**[C05-M07-L01 · Slice analysis](https://learning.nextia-ai.com/courses/ml/m07/slice-analysis/)** · C05, Machine Learning: From Problem to Reliable Model · Module 7, lesson 1 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** sort the model's errors into kinds, compare its results across teams, channels, segments, priorities and weeks, and show the number of tickets next to every rate, so that a small group does not mislead you.

| | |
|---|---|
| **Time** | About 55 minutes |
| **Needs** | An internet connection for the setup cells. No account. |
| **You should know** | The final model and the threshold 0.19, from [Thresholds and calibration](https://learning.nextia-ai.com/courses/ml/m06/thresholds-and-calibration/). Precision and recall, from [Classification metrics](https://learning.nextia-ai.com/courses/ml/m03/classification-metrics/#precision-and-recall). |
| **You do not need** | The local project. The setup below downloads the data and the course's `ticket_model.py`. |
| **Tested** | 2026-10-07, Python 3.14.6 with pandas 3.0.6, scikit-learn 1.9.1 and matplotlib 3.11.2, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/ml/m07/slice-analysis/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C05/M07-L01-slice-analysis/slice-analysis-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs pandas 3.0.6 and scikit-learn 1.9.1, the versions that the course uses, and matplotlib for the charts. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6` and `scikit-learn 1.9.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6 scikit-learn==1.9.1 matplotlib
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the practice data

Run the next cell. It downloads the C05 practice data (about 2.5 MB) into a folder `ticket-model/data`, and checks each file's checksum. It is the same data that `get_data.py` downloads in [Define a baseline](https://learning.nextia-ai.com/courses/ml/m02/define-a-baseline/). It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 16939 rows (1983 escalated), valid 1179 (183), test 1002 (157).` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-model` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course; it has no real people.

The cell also downloads the course's finished project files `ticket_model.py` from [C05/project](https://github.com/usmanahmed99/nextia-learning-labs/tree/main/C05/project) into `ticket-model/`. It is the final version from Select useful features (Module 6), with 11 features. The next cells import it.

In [ ]:
# Setup: download the C05 data into ticket-model/data/ and check it.
# Standard library only: the same steps as get_data.py.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C05/"
FILES = {
    "train.csv": "63b8732a28e891189cc28a9bcd505a093a9c4c51c01f3d0e1fba6d4959242d53",
    "valid.csv": "3604df83ea2ec3cec70c2eedfc7e95cc6d5cb82ccfc3b5cf565e614783e2bde2",
    "test.csv": "60e89f75e4d7d59ccfbda49ca719ce7f8e8d97c0380256d633ab240d546771ec",
    "extra_features.csv": "03c6da28bcb347982aa70a7c72b869771a6cb055654d629da924512c6fd56563",
    "daily_tickets.csv": "7deba76b112b4133a5d16b74be7e2b331ce28c7064e7fd8dd471768e5f1ba360",
    "july_tickets.csv": "10dffc093993341a361172d0aa6ea0e7c06521f140cc2a80ee90c713361f7c99",
    "july_labels.csv": "3163c57e398db6fd1c19e505548d0fa1464821ef96c77901a759abbd3a8605fa",
}
PROJECT_FILES = ["ticket_model.py"]
HERE = Path.cwd()
# The notebook works in its own folder ticket-model/, marked with a file, so
# it never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("ticket-model").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-model and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)
for name in PROJECT_FILES:
    fetch(name, LABS + "project/" + name, PROJECT / name)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
counts = {}
for part in ("train", "valid", "test"):
    rows = Path(f"data/{part}.csv").read_text(encoding="utf-8").splitlines()[1:]
    counts[part] = (len(rows), sum(row.endswith(",1") for row in rows))
print("Ready: train {} rows ({} escalated), valid {} ({}), test {} ({}).".format(
    *counts["train"], *counts["valid"], *counts["test"]))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd` and numpy as `np`, and sets pandas to show 4 decimals. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. One number hides where the errors are

A restaurant with an average review of 4.5 stars can still be bad for you, if all the bad reviews are about the dish you want. An average mixes groups. A model's overall score does the same.

A **slice** is a part of the data chosen by one property, for example all payment tickets. **Slice analysis** computes the same metric for each slice and compares them. It answers the question "Does the model work for everyone it serves?"

Start with 12 made-up tickets from two teams. `flag` is the model's decision, and `escalated_72h` is the truth.

> **Predict.** The model catches 3 of the 6 escalations. Is the recall 50% for each team? Then run the cell.

In [ ]:
tiny = pd.DataFrame({
    "team": ["payment"] * 8 + ["account"] * 4,
    "escalated_72h": [1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0],
    "flag":          [1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0],
})
tiny["caught"] = tiny["flag"] * tiny["escalated_72h"]
print("recall, all tickets:", tiny["caught"].sum() / tiny["escalated_72h"].sum())
print(tiny.groupby("team")[["escalated_72h", "caught"]].sum())

> **Check.** You should see `recall, all tickets: 0.5`. Payment has 4 escalations and 3 caught (75%). Account has 2 escalations and 0 caught (0%).

The overall 50% is a mix of 75% and 0%. But account has only 2 escalations. One more caught ticket would change its recall from 0% to 50%. So always look at the count next to the rate.

## 2. Score the validation month

The next cell fits the final model on the train set and scores the validation month (May 2026) with the threshold 0.19. Every decision in this module uses the validation set (the data we keep to check and improve the model), never the test set.

> **Check.** You should see `1179 tickets, 227 flagged, 183 escalated`. Fitting takes a few seconds.

In [ ]:
from ticket_model import FEATURES, TARGET, build_pipeline, load

train, valid = load("train"), load("valid")
pipeline = build_pipeline().fit(train[FEATURES], train[TARGET])
threshold = 0.19
valid["score"] = pipeline.predict_proba(valid[FEATURES])[:, 1]
valid["flag"] = (valid["score"] >= threshold).astype(int)
print(len(valid), "tickets,", valid["flag"].sum(), "flagged,", valid[TARGET].sum(), "escalated")

## 3. Four kinds of error

Not every error is the same. The next cell gives each ticket an **outcome**:

- **caught**: flagged and escalated.
- **false alarm**: flagged, but not escalated. It costs about 10 minutes of a senior agent.
- **missed, near threshold**: escalated, not flagged, with a score from 0.10 to just under 0.19. A slightly lower threshold could catch it.
- **missed, low score**: escalated, with a score below 0.10. The model saw no risk at all.

The [diagram on the lesson page](https://learning.nextia-ai.com/courses/ml/m07/slice-analysis/#four-kinds-of-error) shows the same tree.

> **Predict.** Of the 97 missed escalations, are most near the threshold or far below it? Then run the cell.

In [ ]:
escalated = valid[TARGET] == 1
flagged = valid["flag"] == 1
valid["outcome"] = "correct pass"
valid.loc[flagged & escalated, "outcome"] = "caught"
valid.loc[flagged & ~escalated, "outcome"] = "false alarm"
valid.loc[~flagged & escalated & (valid["score"] >= 0.10), "outcome"] = "missed, near threshold"
valid.loc[~flagged & escalated & (valid["score"] < 0.10), "outcome"] = "missed, low score"
print(valid["outcome"].value_counts())

> **Check.** You should see `correct pass 855`, `false alarm 141`, `caught 86`, `missed, low score 55` and `missed, near threshold 42`.

55 of the 97 missed escalations have a score below 0.10. No threshold that fits the capacity can catch them.

Two more questions apply to every error. **Data problem:** is an input unknown or missing? **Label problem:** does the target mean what we think? The next cell marks tickets with a data problem: `channel` or `region` is `unknown`, or `order_value` is missing outside the account team. `pd.crosstab` counts the tickets for each pair of values.

In [ ]:
valid["data_problem"] = ((valid["channel"] == "unknown") | (valid["region"] == "unknown")
                         | (valid["order_value"].isna() & (valid["team"] != "account")))
print(pd.crosstab(valid["outcome"], valid["data_problem"]))

> **Check.** You should see 90 tickets in the `True` column: 6 caught, 7 false alarms, 5 missed with a low score, 2 missed near the threshold and 70 correct passes.

Only 14 of the 238 errors have a data problem. That is 6%, less than the 8% of all tickets (90 of 1,179). So data problems do not explain most errors here. Look for the cause elsewhere.

## 4. Slices with counts

The next cell defines `slice_table()`. For each value of a column, it counts the tickets, escalations, flags and caught escalations. Then it computes the escalation rate (`esc_rate`), the recall, the precision and the mean score. It shows the slices by team.

> **Predict.** Which team has the lowest recall? Then run the cell.

In [ ]:
def slice_table(df, column):
    """Tickets, escalations, flags and recall for each value of one column."""
    df = df.assign(caught=df["flag"] * df[TARGET])
    table = df.groupby(column).agg(tickets=(TARGET, "size"), escalated=(TARGET, "sum"),
                                   flagged=("flag", "sum"), caught=("caught", "sum"),
                                   mean_score=("score", "mean"))
    table["esc_rate"] = table["escalated"] / table["tickets"]
    table["recall"] = table["caught"] / table["escalated"]
    table["precision"] = table["caught"] / table["flagged"]
    return table.round(3)


slice_table(valid, "team")

> **Check.** You should see five rows. Payment has recall 0.771 (37 of 48). Account has 0.083 (1 of 12).

Read two more columns:

- **Account** has 12 escalations. Its recall, 1 of 12, is uncertain, but it is low in every way you look at it: the mean score is 0.057.
- **Warranty** has an escalation rate of 0.286, but a mean score of only 0.163. In the train set, warranty's rate was 0.159. The new warranty policy of 1 May changed what "escalated" means for warranty tickets. That is a **label problem**: the model learned the old meaning.

## 5. A small slice

Run the next cell. It makes the slice table for `channel`.

> **Check.** You should see the `unknown` channel with 22 tickets, 5 escalated, 3 caught and recall 0.600.

In [ ]:
slice_table(valid, "channel")

The `unknown` channel has the best recall, but it has only 22 tickets in May. The next cell computes its escalation rate with its **standard error** (SE), the typical size of the chance error, as in C03. Then it adds and removes two escalations.

> **Predict.** With 22 tickets, how far can the rate move if two tickets had a different outcome? Then run the cell.

In [ ]:
unknown = valid[valid["channel"] == "unknown"]
k, n = int(unknown[TARGET].sum()), len(unknown)
p = k / n
se = (p * (1 - p) / n) ** 0.5
print(f"{k} of {n} escalated: rate {p:.3f}, SE {se:.3f}, 95% interval {p - 1.96 * se:.3f} to {p + 1.96 * se:.3f}")
print("two escalations fewer or more:", round((k - 2) / n, 3), round((k + 2) / n, 3))

> **Check.** You should see `5 of 22 escalated: rate 0.227, SE 0.089, 95% interval 0.052 to 0.402`, then `0.136 0.318`.

Two tickets move the rate by 9 points in each direction. For email (467 tickets), the SE is 0.017, and two tickets move the rate by less than half a point. A rate from about 20 tickets can only tell you "somewhere from 5% to 40%".

## 6. Guided practice: slices by priority

> **Your turn.** Make the slice table for `priority`. Put it in `by_priority`. Then put the recall of priority 3 in `recall_p3`, rounded to 3 decimals. Run the cell, then the check cell.

In [ ]:
by_priority = slice_table(valid, "priority")
recall_p3 = round(float(by_priority.loc[3, "recall"]), 3)
by_priority

In [ ]:
expect_hash('the recall of priority 3', recall_p3, '0e00258e73f3ae8b')

Answer three questions in the next cell (double-click to edit). How many escalations does each priority have? Where are most of the missed escalations? Is the slice large enough to trust?

*Your answers here:*

## 7. Change one thing: slice by week

Change the column: slice by time instead of by group. A week of May here is days 1–7, 8–14, 15–21, 22–28, and 29–31 (only three days).

> **Predict.** Will the recall be about the same each week? Which week has the fewest tickets?

> **Your turn.** Make a column `week` with `(valid["created_at"].dt.day - 1) // 7 + 1`. Then put the slice table for `week` in `by_week`. Run the cell, then the check cell.

In [ ]:
valid["week"] = (valid["created_at"].dt.day - 1) // 7 + 1
by_week = slice_table(valid, "week")
by_week

In [ ]:
expect_hash('the tickets per week', list(by_week["tickets"]) if hasattr(by_week, "columns") else by_week, '99736da2d9df4120')

Compare the recall per week with the count of escalations. Week 5 has only 9 escalations: one ticket moves its recall by 11 points. The [lesson page](https://learning.nextia-ai.com/courses/ml/m07/slice-analysis/#your-turn-slice-by-week) has the model answer.

## 8. Failure case: acting on a test slice

Tomás looks at the test month (June) for the final report. The next cell makes the same slice table on the test set.

> **Warning.** The test set is for the final report only. Report what you see; do not change the model, the threshold or a rule because of it.

In [ ]:
test = load("test")
test["score"] = pipeline.predict_proba(test[FEATURES])[:, 1]
test["flag"] = (test["score"] >= threshold).astype(int)
slice_table(test, "channel")

> **Check.** You should see the `unknown` channel with 23 tickets, 6 escalated and 0 caught.

Tomás says: "The model fails on unknown-channel tickets. Let us flag all of them." Two things are wrong. Six escalations are too few: in May, the same slice caught 3 of 5. And a change made because of the test set turns the test set into a second validation set, so the final score is no longer honest. The fix: write it in the report as a slice to watch ("0 of 6, too few to judge"), and decide with more validation data. The [lesson page](https://learning.nextia-ai.com/courses/ml/m07/slice-analysis/#failure-case-acting-on-a-test-slice) has the full case.

## Check your understanding and recap

Answer the **knowledge checks** on the [lesson page](https://learning.nextia-ai.com/courses/ml/m07/slice-analysis/#knowledge-check). They count toward your certificate when you are signed in and enrolled.

In this lesson you sorted the 238 errors of May into kinds, and found that 55 missed escalations had a score below 0.10. The overall recall, 0.470, hides 0.771 for payment, 0.083 for account and 0.102 for priority 3. Warranty escalates far more than its scores say, because the policy changed. A rate from 22 tickets moves 9 points when two tickets change, so the count goes next to every rate. Next, you look inside the model: which features push a score up or down.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Explain predictions](https://learning.nextia-ai.com/courses/ml/m07/explain-predictions/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/ml/m07/slice-analysis/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.